# STATISTIQUES

## IMPORTS

In [37]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as spark_func

## Création de la session Spark et récupération du dataset

In [38]:
path = "data/gaming_mental_health_merged.csv"
#"hdfs://localhost:9000/projet/gaming_mental_health/input/gaming_mental_health.csv"

spark = (
    SparkSession.builder
    .appName("MonPremierProjet")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

## Création des différentes statistiques

### Âge moyen des personnes du dataset

In [ ]:
mean_age = (df.select(df.age)
            .agg(spark_func.round(spark_func.mean(df.age),1).alias("Average age"))
)

print("Average age in the dataset :")
mean_age.show()

Average age in the dataset :
+--------+
|Mean age|
+--------+
|    20.6|
+--------+



### Proportion sexes par genre de jeu

In [43]:
gender_count_by_game_genre = df.groupBy(df.game_genre, df.gender).count()

people_by_game_genre = gender_count_by_game_genre.groupBy(df.game_genre).agg(spark_func.sum("count").alias("total"))

result1 = (
    gender_count_by_game_genre.join(people_by_game_genre, on="game_genre")
          .withColumn("proportion(%)", spark_func.round((spark_func.col("count") / spark_func.col("total"))*100, 1))
)

print("Female/Male/Other proportions depending on game genres :")
result1.orderBy("game_genre", "gender").show(result1.count())

Female/Male/Other proportions depending on game genres :
+-------------+------+-----+-----+-------------+
|   game_genre|gender|count|total|proportion(%)|
+-------------+------+-----+-----+-------------+
|Battle Royale|Female|   57|  142|         40.1|
|Battle Royale|  Male|   83|  142|         58.5|
|Battle Royale| Other|    2|  142|          1.4|
|          FPS|Female|   39|  139|         28.1|
|          FPS|  Male|   97|  139|         69.8|
|          FPS| Other|    3|  139|          2.2|
|          MMO|Female|   38|  143|         26.6|
|          MMO|  Male|  101|  143|         70.6|
|          MMO| Other|    4|  143|          2.8|
|         MOBA|Female|   52|  160|         32.5|
|         MOBA|  Male|  103|  160|         64.4|
|         MOBA| Other|    5|  160|          3.1|
| Mobile Games|Female|   52|  142|         36.6|
| Mobile Games|  Male|   88|  142|         62.0|
| Mobile Games| Other|    2|  142|          1.4|
|          RPG|Female|   47|  159|         29.6|
|          R

In [45]:
print("Genres where women are most present (top 5) :")
result2 = result1.select("game_genre", "gender", "proportion(%)").filter(result1.gender=="Female")
result2.sort("proportion(%)",ascending = False).show(5)

Genres where women are most present (top 5) :
+-------------+------+-------------+
|   game_genre|gender|proportion(%)|
+-------------+------+-------------+
|Battle Royale|Female|         40.1|
| Mobile Games|Female|         36.6|
|     Strategy|Female|         33.3|
|         MOBA|Female|         32.5|
|          RPG|Female|         29.6|
+-------------+------+-------------+
only showing top 5 rows


### Dépenses moyennes par sexe

In [53]:
mean_spending_per_gender = (
    df.select(df.gender, df.monthly_game_spending_usd)
    .groupBy(df.gender)
    .agg(spark_func.round(spark_func.avg(df.monthly_game_spending_usd),2).alias("Mean spending per month"))
)

print("Mean spending on video games per month per gender :")
mean_spending_per_gender.show()

Mean spending on video games per month per gender :
+------+-----------------------+
|gender|Mean spending per month|
+------+-----------------------+
|Female|                  97.64|
| Other|                 111.12|
|  Male|                 105.04|
+------+-----------------------+



### Moyenne d'heure de sommeil par jeu

In [65]:
mean_sleep_hours_per_game = (
    df.select(df.primary_game, df.sleep_hours)
    .groupBy(df.primary_game)
    .agg(spark_func.round(spark_func.avg(df.sleep_hours),2).alias("Average sleep hours"))
)

print("Average hours of sleep per main game played :")
mean_sleep_hours_per_game.sort("Average sleep hours").show(mean_sleep_hours_per_game.count())

Average hours of sleep per main game played :
+--------------------+-------------------+
|        primary_game|Average sleep hours|
+--------------------+-------------------+
|    Honkai star rail|                4.0|
|         PUBG Mobile|               5.32|
|     Civilization VI|               5.33|
|          Elden Ring|               5.46|
|            Fortnite|               5.49|
|      Cyberpunk 2077|               5.56|
|        Apex Legends|               5.59|
|              Dota 2|                5.6|
|   Final Fantasy XIV|               5.61|
|        StarCraft II|               5.61|
|      Clash of Clans|               5.66|
|   League of Legends|               5.67|
|        Call of Duty|               5.76|
|   World of Warcraft|               5.79|
|               CS:GO|               5.79|
|      Age of Empires|               5.89|
|                PUBG|               5.91|
|Elder Scrolls Online|               5.92|
|      Genshin Impact|               5.94|
|       

### Plus vieilles bases de joueurs

In [ ]:
oldest_playerbase = (df.select(df.primary_game, df.age)
                    .groupBy(df.primary_game)
                    .agg(spark_func.round(spark_func.avg(df.age),2).alias("Average age"))
)

print("Top 5 oldest playerbases :")
oldest_playerbase.sort("Average age", ascending = False).show(5)

top 5 oldest playerbases :
+--------------------+-----------+
|        primary_game|Average age|
+--------------------+-----------+
|           Everquest|       43.0|
|Just finished Nie...|       34.0|
|           Minecraft|       28.0|
|     Path of Exile 2|       26.0|
|            Among us|       24.0|
+--------------------+-----------+
only showing top 5 rows
